### Here we will create a Data pipeline for BERT with Masked Language Modeling (MLM)

In [2]:
import torch
import random

from tokenizers import Tokenizer

BATCH_SIZE = 16

BLOCK_SIZE = 128

MASK_PROB = 0.15

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [3]:
tokenizer = Tokenizer.from_file(
    "tokenizer.json"
)

print(
    tokenizer.get_vocab_size()
)

5000


In [4]:
with open(
    "tiny_shakespeare.txt",
    "r",
    encoding="utf-8"
) as f:

    text = f.read()

In [5]:
encoded = tokenizer.encode(
    text
)

token_ids = encoded.ids

In [6]:
data = torch.tensor(
    token_ids,
    dtype=torch.long
)

In [7]:
n = int(
    0.9 * len(data)
)

train_data = data[:n]

val_data = data[n:]

In [8]:
vocab = tokenizer.get_vocab()

print(
    "[MASK]" in vocab
)

False


In [9]:
MASK_TOKEN_ID = 4

### Createing MLM function

In [10]:
def mask_tokens(
    inputs
):

    labels = inputs.clone()
    #Create random matrix
    probability_matrix = torch.full(
        labels.shape,
        MASK_PROB
    )
    # Generate mask positions
    masked_indices = (
        torch.bernoulli(
            probability_matrix
        )
        .bool()
    )

    labels[
        ~masked_indices
    ] = -100

    inputs[
        masked_indices
    ] = MASK_TOKEN_ID

    return inputs, labels

In [11]:
# Test MLM
sample = train_data[
    :20
].clone()

print(sample)

masked, labels = mask_tokens(
    sample.clone()
)

print(masked)

print(labels)

tensor([ 407,  765,   12, 1975,  116, 2271,  424, 1915,    8,  395,   81,  365,
          10,  882,   12, 2030,    8,  365,   10,  407])
tensor([ 407,  765,   12, 1975,  116,    4,  424, 1915,    8,  395,   81,  365,
          10,  882,   12,    4,    4,  365,   10,  407])
tensor([-100, -100, -100, -100, -100, 2271, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, 2030,    8, -100, -100, -100])


In [12]:
def get_batch(
    split
):

    source = (
        train_data
        if split == "train"
        else val_data
    )

    ix = torch.randint(

        len(source)
        -
        BLOCK_SIZE
        -
        1,

        (BATCH_SIZE,)
    )

    x = torch.stack(

        [

            source[
                i:i+BLOCK_SIZE
            ]

            for i in ix

        ]

    )

    masked_x, labels = mask_tokens(
        x.clone()
    )

    return masked_x, labels

In [13]:
# Test Batch
xb, labels = get_batch(
    "train"
)

print(
    xb.shape
)

print(
    labels.shape
)

torch.Size([16, 128])
torch.Size([16, 128])


In [14]:
# Count of Masked tokens
num_masked = (
    labels != -100
).sum()

print(
    num_masked
)

tensor(301)


In [15]:
xb, labels = get_batch(
    "train"
)

xb = xb.to(device)

labels = labels.to(device)

print(
    xb.device
)

cuda:0


### What i have done here
Tiny Shakespeare -> Tokenizer -> Token IDs -> Random Context Windows -> Mask 15% Tokens -> Input:[MASK] tokens ->
Labels:
Original tokens